# Jour 1 · Mini-projet : choisir un modèle honnêtement

## Objectifs

- comparer plusieurs modèles avec le même protocole
- choisir sur la validation puis évaluer une seule fois sur le test
- formuler une recommandation compréhensible par le métier


## Mission

Vous devez proposer un premier détecteur de risque de panne. Trois candidats sont disponibles : régression logistique, arbre de décision et forêt aléatoire. Nous utiliserons la validation pour comparer les candidats. Le test ne sera ouvert qu'après le choix.

### Les trois candidats

- La **régression logistique** construit une frontière simple et reste facile à expliquer.
- Un **arbre de décision** enchaîne des questions du type « vibration supérieure à une valeur ? ». `max_depth=4` limite ici le nombre de niveaux pour éviter un arbre trop complexe.
- Une **forêt aléatoire** combine plusieurs arbres légèrement différents puis agrège leurs décisions. `n_estimators=150` demande 150 arbres ; `n_jobs=-1` autorise l'utilisation des cœurs CPU disponibles pour accélérer le calcul.

Ces paramètres sont des **hyperparamètres** : ils sont choisis avant l'entraînement, contrairement aux paramètres appris automatiquement par le modèle.



![Comparaison intuitive de trois familles de modèles](../ressources/illustrations/jour_01/04_trois_modeles.png)

La régression logistique combine toutes les features dans un score puis le transforme en probabilité. Sa frontière de décision reste relativement simple. Les coefficients peuvent aider à comprendre le sens d’une relation, à condition que les données aient été correctement préparées.

L’arbre de décision découpe progressivement les observations. Chaque nœud pose une question, chaque branche représente une réponse et chaque feuille produit une décision. Un arbre profond peut créer des règles très spécifiques aux données d’entraînement.

La forêt aléatoire entraîne de nombreux arbres sur des échantillons et des sous-ensembles de features légèrement différents. Le vote de plusieurs modèles réduit souvent l’instabilité d’un arbre isolé. En contrepartie, expliquer chaque décision devient moins direct.

Aucune famille n’est toujours gagnante. Nous devons les comparer avec le même découpage, la même cible et les mêmes métriques.


![Comparaison de quatre niveaux de modèles](../ressources/illustrations/jour_01/04_comparer_modeles.png)

*Une comparaison honnête conserve exactement les mêmes données et métriques pour tous les candidats.*


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_course_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ressources" / "donnees" / "maintenance_machines.csv").exists():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable. Ouvrez le notebook depuis le dossier du cours.")


ROOT = find_course_root()
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"
plt.style.use("seaborn-v0_8-whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv(DATA_FILE)
features = ["temperature_c", "vibration_mm_s", "pressure_bar", "load_pct", "age_years", "days_since_maintenance"]
X, y = df[features], df["failure_7d"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=.40, stratify=y, random_state=42)
X_validation, X_test, y_validation, y_test = train_test_split(X_temp, y_temp, test_size=.50, stratify=y_temp, random_state=42)

candidates = {
    "logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    "arbre": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "forêt": RandomForestClassifier(n_estimators=150, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1),
}


## Complexité, sous-apprentissage et surapprentissage

![Équilibre entre sous-apprentissage, bonne généralisation et surapprentissage](../ressources/illustrations/jour_01/04_complexite_generalisation.png)

Un modèle trop simple peut manquer des relations utiles : c'est le **sous-apprentissage**. Un modèle trop complexe peut mémoriser les particularités du jeu d'entraînement : c'est le **surapprentissage**.

Ce que nous recherchons est la **généralisation** : de bonnes prédictions sur des exemples jamais vus. Pour la mesurer honnêtement, on compare les scores d'entraînement et de validation.


In [ ]:
profondeurs = [1, 2, 3, 4, 6, 10, None]
lignes_complexite = []

for profondeur in profondeurs:
    arbre_test = DecisionTreeClassifier(
        max_depth=profondeur,
        class_weight="balanced",
        random_state=42,
    )
    arbre_test.fit(X_train, y_train)
    lignes_complexite.append({
        "profondeur": "sans limite" if profondeur is None else str(profondeur),
        "F1 entraînement": f1_score(y_train, arbre_test.predict(X_train)),
        "F1 validation": f1_score(y_validation, arbre_test.predict(X_validation)),
    })

complexite = pd.DataFrame(lignes_complexite)
display(complexite.round(3))

positions = np.arange(len(complexite))
plt.figure(figsize=(8, 4))
plt.plot(positions, complexite["F1 entraînement"], marker="o", label="entraînement")
plt.plot(positions, complexite["F1 validation"], marker="o", label="validation")
plt.xticks(positions, complexite["profondeur"])
plt.xlabel("max_depth de l'arbre")
plt.ylabel("score F1")
plt.title("La performance d'entraînement ne suffit pas")
plt.ylim(0, 1.02)
plt.legend()
plt.show()


### Lire les deux courbes de complexité

Le score d’entraînement augmente généralement avec la profondeur : un arbre plus grand possède davantage de possibilités pour reproduire les exemples vus.

Le score de validation est le véritable contrôle. S’il reste faible avec un arbre très simple, le modèle sous-apprend. S’il baisse alors que le train continue de progresser, l’écart entre les deux courbes signale un surapprentissage probable.

La meilleure profondeur n’est pas forcément celle qui produit le maximum absolu sur un seul découpage. On recherche une zone où la validation reste bonne et où le modèle n’est pas inutilement complexe.


### Hyperparamètre et validation croisée

`max_depth` est un **hyperparamètre** : une décision prise avant l'entraînement qui contrôle ici la complexité de l'arbre. Les poids d'un réseau ou les séparations d'un arbre sont au contraire des **paramètres appris** à partir des données.

Une seule validation peut dépendre d'un découpage chanceux. La **validation croisée** recommence l'entraînement sur plusieurs découpages et moyenne les résultats. Des outils comme `GridSearchCV` testent une liste de réglages ; `RandomizedSearchCV` en essaie un échantillon. Leur rôle est d'organiser la comparaison, pas de remplacer le raisonnement métier ni le jeu de test final.


![Principe de la validation croisée](../ressources/illustrations/jour_01/04_validation_croisee.png)

Dans une validation croisée à quatre plis, les données disponibles sont divisées en quatre blocs. Le modèle est entraîné quatre fois : à chaque tour, trois blocs servent au train et le dernier à la validation. On moyenne ensuite les quatre scores.

Cela donne une estimation moins dépendante d’un seul tirage, mais coûte davantage de temps car l’entraînement est répété. Toutes les étapes de préparation doivent rester à l’intérieur de chaque pli, ce que facilite une Pipeline.

Le test final ne fait pas partie de ces plis. Il reste à l’écart pendant le choix de la famille de modèle et des hyperparamètres.


In [ ]:
def evaluate(truth, predictions):
    return {
        "precision": precision_score(truth, predictions, zero_division=0),
        "recall": recall_score(truth, predictions, zero_division=0),
        "f1": f1_score(truth, predictions, zero_division=0),
        "alertes": int(predictions.sum()),
    }

validation_rows = []
for name, candidate in candidates.items():
    candidate.fit(X_train, y_train)
    validation_rows.append({"modèle": name, **evaluate(y_validation, candidate.predict(X_validation))})
validation_results = pd.DataFrame(validation_rows).set_index("modèle")
validation_results.round(3)


## Lire la table de comparaison

Chaque ligne correspond à un candidat entraîné sur exactement `X_train`. Les prédictions sont ensuite produites sur `X_validation`.

- `precision` mesure la qualité des alertes ;
- `recall` mesure la proportion de pannes retrouvées ;
- `f1` résume ces deux dimensions ;
- `alertes` rend visible la charge opérationnelle.

Le modèle choisi ici maximise F1 pour fournir une règle reproductible. Une entreprise pourrait retenir un autre compromis si le coût des pannes manquées ou des inspections inutiles l’exige.


## Choisir ne signifie pas chercher le score le plus grand partout

Une équipe peut préférer un rappel élevé si manquer une panne coûte très cher, ou une meilleure précision si chaque alerte déclenche une intervention lourde. Le nombre d'alertes complète donc les métriques.

Une fois le modèle choisi sur la validation, nous le réentraînons sur train + validation, puis nous ouvrons le test une seule fois.


In [ ]:
selected_name = validation_results["f1"].idxmax()
selected_model = candidates[selected_name]
X_train_final = pd.concat([X_train, X_validation])
y_train_final = pd.concat([y_train, y_validation])
selected_model.fit(X_train_final, y_train_final)
test_predictions = selected_model.predict(X_test)
print("Modèle choisi :", selected_name)
pd.Series(evaluate(y_test, test_predictions), name="test final").round(3)


## Pourquoi réentraîner avant le test final ?

Une fois le candidat choisi, la validation n’est plus nécessaire pour comparer. On rassemble donc train et validation afin de fournir davantage d’exemples au modèle retenu, puis on l’entraîne à nouveau depuis le début.

Le test est ouvert une seule fois pour estimer le résultat final. Si ce score nous conduit à changer de modèle ou d’hyperparamètre, le test devient de fait une nouvelle validation et n’est plus un contrôle indépendant.

Un résultat final doit toujours être accompagné du protocole, du volume de données, des métriques et des limites observées.


## À vous de jouer — rédiger la conclusion

Affichez la ligne du meilleur modèle sur la validation puis écrivez deux phrases : une force observée et une limite à vérifier avant production.


In [ ]:
# Affichez le résultat puis rédigez votre conclusion dans deux chaînes.
pass


## Bilan du Jour 1

Vous savez maintenant traduire une question en `X` et `y`, découper les données, entraîner avec `fit`, produire des résultats avec `predict` et lire les erreurs au-delà de l'accuracy.

Demain, nous conserverons exactement ces réflexes. Le réseau de neurones changera la manière d'apprendre les représentations, mais pas la nécessité d'un protocole honnête.
